# VisionGuard AI - Milestone 01: Dataset Exploration & Integrity Check

**Objective:**
1. Inspect and validate the MVTec AD dataset structure for the target categories: `metal_nut`, `screw`, and `capsule`.
2. Visualize normal training samples vs. various defect categories.
3. Inspect pixel-level ground truth masks to understand defect morphology.
4. Assess resolution, color space, and intensity distribution to inform our preprocessing pipeline.

In [ ]:
import os
from pathlib import Path
import yaml
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image

# Change directory to project root if running from notebooks directory
if Path.cwd().name == "notebooks":
    os.chdir("..")
print(f"Current working directory: {Path.cwd()}")

# Load configuration
with open("configs/data.yaml", "r") as f:
    config = yaml.safe_load(f)

RAW_DATA_DIR = Path(config["dataset"]["raw_dir"])
CATEGORIES = config["dataset"]["target_categories"]
print(f"Target Categories: {CATEGORIES}")

## 1. Run Automated Integrity Verification

In [ ]:
from src.data.verify_dataset import run_verification

is_valid = run_verification(str(RAW_DATA_DIR))
assert is_valid, "Please check missing folders or masks before proceeding."

## 2. Inspect Image Dimensions and Properties

In [ ]:
for cat in CATEGORIES:
    sample_path = next((RAW_DATA_DIR / cat / "train" / "good").glob("*.png"))
    with Image.open(sample_path) as img:
        print(f"[{cat}] Sample: {sample_path.name} | Size: {img.size} | Mode: {img.mode}")

## 3. Visualize Normal Samples vs Anomalous Samples with Ground-Truth Masks

In [ ]:
def visualize_category_samples(category: str, num_samples_per_defect: int = 1):
    test_dir = RAW_DATA_DIR / category / "test"
    gt_dir = RAW_DATA_DIR / category / "ground_truth"
    
    defect_types = [d.name for d in sorted(test_dir.iterdir()) if d.is_dir() and d.name != "good"]
    n_defects = len(defect_types)
    
    fig, axes = plt.subplots(n_defects, 3, figsize=(12, 3.5 * n_defects))
    if n_defects == 1:
        axes = np.expand_dims(axes, 0)
        
    for i, defect in enumerate(defect_types):
        img_file = next((test_dir / defect).glob("*.png"))
        mask_file = gt_dir / defect / f"{img_file.stem}_mask.png"
        
        img = cv2.imread(str(img_file))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(str(mask_file), cv2.IMREAD_GRAYSCALE)
        
        # Create overlay
        overlay = img.copy()
        overlay[mask > 0] = [255, 0, 0] # Highlight defect region in red
        blended = cv2.addWeighted(img, 0.7, overlay, 0.3, 0)
        
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"{category} - {defect} (Original)")
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(mask, cmap='gray')
        axes[i, 1].set_title(f"Ground Truth Mask ({defect})")
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(blended)
        axes[i, 2].set_title(f"Overlay (Defect Marked)")
        axes[i, 2].axis('off')
        
    plt.tight_layout()
    plt.show()

# Visualize for metal_nut
visualize_category_samples("metal_nut")